# GVH Diagonal Cubic — CHH-07
## Physical source and nonzero-shear partial Euler–Lagrange gate

**État :** audit complémentaire, non canonique. Aucune nouvelle validation astrophysique.

**Mission :** partir de l'action archivée `S_cand`, ajouter **explicitement** un secteur matière covariant de contrôle, vérifier ses EOM et son tenseur énergie-impulsion, construire un témoin **nonrigide à cisaillement non nul**, tester le trièdre et une **branche vectorielle limitée**. Vérifier séparément le verrou de l'équation métrique.

**Règle :** un PASS sur l'équation scalaire, le trièdre ou une sous-branche vectorielle ne signifie **pas** solution des EOM complètes de `(g,u,e,φ)`, ni interaction inter-cube ou prédiction GVH indépendante.

### Nos quatre niveaux obligatoires
1. **GVH :** conserver l'action originale, isoler hypothèses, contraintes et dérivations.
2. **Physique établie :** champ de Klein–Gordon, tenseur énergie-impulsion, bilans locaux.
3. **Test numérique minimal :** démonstrations symboliques, énergie sur une période, résidus et branches.
4. **Verdict scientifique :** PASS local / BLOCKED pour les EOM gravitationnelles intégrales.

**Unités :** coordonnées `t,x` et paramètres du témoin sont **adimensionnés** et `c=ℏ=1` par convention mathématique ; aucune échelle absolue SI n'est déterminée ici. Données ALMA non chargées.

## Niveau 1 — Action candidate réellement archivée

Signature `(-,+,+,+)`, champ unitaire `u^μu_μ=-1`, trièdre `u·e_A=0`, `e_A·e_B=δ_AB`.

\[
S_{\rm cand}=\frac{1}{16\pi G_0}\int d^4x\sqrt{-g}
\left[R+\mathcal L_u+\lambda_u(u^2+1)+\frac{\zeta_C}{2}\mathcal O_C
+\sum_A\eta_A (u\cdot e_A)+\frac12\sum_{AB}\eta_{AB}(e_A\cdot e_B-\delta_{AB})\right].
\]

\[
\mathcal L_u=-c_1(\nabla_\mu u_\nu)(\nabla^\mu u^\nu)-c_2(\nabla_\mu u^\mu)^2
-c_3(\nabla_\mu u_\nu)(\nabla^\nu u^\mu)+c_4 a_\mu a^\mu,
\quad a^\mu=u^\nu\nabla_\nu u^\mu.
\]

\[
\mathcal O_C=\sum_{A=1}^3 (e_A^\mu e_A^\nu \sigma_{\mu\nu})^2.
\]

**Important :** `S_cand` archivée ne contient ni la matière de ce notebook, ni une variable `A_C`, ni un secteur physique de transport entre cubes. Le trièdre n'a pas de cinétique propre `∇e` dans cette action. Le terme cubique et les contraintes sont des composants du **candidat GVH**, non une nouvelle loi validée.

In [1]:
import json, hashlib
from pathlib import Path
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import simpson

NAME = 'GVH_CHH_07_Physical_Source_and_Nonzero_Shear_EOM_Gate_FAST'
CHECKS = {}
def check(label, condition):
    CHECKS[label] = bool(condition)
    assert CHECKS[label], 'FAILED: '+label

SOURCE_PROVENANCE = {'source_28': {'basename': 'GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28_Full_GVH_EOM_Derivation_and_Variational_Closure_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb', 'sha256': 'c8015f1d771caa5f65c8ad5d64505733ebfa96a81f6153a8b1d8c2233cfbeefa'}, 'source_28_1': {'basename': 'GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.1_Cubic_Vector_Closure_Metric_Helix_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb', 'sha256': '0018b54c8f07984e3ebf35ea6989e6772bebb815492b66b672bed83241f8d5b4'}, 'chh05_results': {'basename': 'GVH_CHH_05_Action_Coupling_and_EOM_Gate_FAST_results.json', 'sha256': '3c29c0b61682e185f261fb80fb66c98e3fcaab144a93b51c196831ef8976ea71'}, 'chh06_results': {'basename': 'GVH_CHH_06_Independent_Observable_and_SI_Source_Gate_FAST_results.json', 'sha256': '8761fa5e678540db6a19bd11249528cb28655172c254581e8a4b83e117eb6027'}}
for k,v in SOURCE_PROVENANCE.items():
    optional = Path('gvh_source')/v['basename'] if k.startswith('source_') else Path(v['basename'])
    if optional.exists():
        actual = hashlib.sha256(optional.read_bytes()).hexdigest()
        check('source_sha_optional_'+k, actual == v['sha256'])
print('Source provenance hashes archived / optionally verified:', SOURCE_PROVENANCE)
print('External sector is NOT part of original S_cand.')

Source provenance hashes archived / optionally verified: {'source_28': {'basename': 'GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28_Full_GVH_EOM_Derivation_and_Variational_Closure_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb', 'sha256': 'c8015f1d771caa5f65c8ad5d64505733ebfa96a81f6153a8b1d8c2233cfbeefa'}, 'source_28_1': {'basename': 'GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.1_Cubic_Vector_Closure_Metric_Helix_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb', 'sha256': '0018b54c8f07984e3ebf35ea6989e6772bebb815492b66b672bed83241f8d5b4'}, 'chh05_results': {'basename': 'GVH_CHH_05_Action_Coupling_and_EOM_Gate_FAST_results.json', 'sha256': '3c29c0b61682e185f261fb80fb66c98e3fcaab144a93b51c196831ef8976ea71'}, 'chh06_results': {'basename': 'GVH_CHH_06_Independent_Observable_and_SI_Source_Gate_FAST_results.json', 'sha256': '8761fa5e678540db6a19bd11249528cb28655172c254581e8a4b83e117eb6027'}}
External sector is NOT part of original S_cand.


## 1.1 — Secteur matière : définition **ajoutée** et dérivation

Pour un champ scalaire réel `φ`, posons **comme extension non canonique**

\[
S_m=-\frac12\int d^4x\sqrt{-g}\,[g^{\mu\nu}\partial_\mu\phi\partial_\nu\phi+\mu^2\phi^2].
\]

Cette action entraîne les équations classiques :

\[
(\Box_g-\mu^2)\phi=0,\qquad
T_{\mu\nu}=\partial_\mu\phi\partial_\nu\phi-
\frac12 g_{\mu\nu}[(\partial\phi)^2+\mu^2\phi^2].
\]

Identité de Noether : \(\nabla_\mu T^{\mu\nu}=(\Box_g\phi-\mu^2\phi)\nabla^\nu\phi\). La conservation locale est donc **sur couche matière** (on shell) et n'assure pas la fermeture métrique.

Dans la sous-limite `c_i=ζ_C=0` et multiplicateurs nuls, la métrique doit satisfaire \(G_{\mu\nu}=8\pi G_0 T_{\mu\nu}\). Une source non nulle ne peut généralement pas être posée sur Minkowski comme **solution couplée exacte**.

In [2]:
t,x = sp.symbols('t x', real=True)
Phi,k,mu = sp.symbols('Phi k mu', positive=True, real=True)
omega=sp.symbols('omega',positive=True,real=True)
phi = Phi*sp.cos(omega*t)*sp.cos(k*x)
phi_t=sp.diff(phi,t)
phi_x=sp.diff(phi,x)
EL=sp.diff(phi,t,2)-sp.diff(phi,x,2)+mu**2*phi
EL_on_shell=sp.simplify(EL.subs(omega**2,k**2+mu**2))
T00=sp.Rational(1,2)*(phi_t**2+phi_x**2+mu**2*phi**2)
T01=-phi_t*phi_x
continuity=sp.simplify(sp.diff(T00,t)+sp.diff(T01,x))
expected=sp.simplify(phi_t*EL)
check('KG_symbolic_shell',EL_on_shell==0)
check('stress_continuity_noether_identity',sp.simplify(continuity-expected)==0)
check('matter_nonnegative_local_energy',float(T00.subs({Phi:.2,k:2,mu:1.25,omega:np.sqrt(4+1.25**2),t:0,x:0}))>0)
print('KG residual =',sp.factor(EL))
print('continuity - phi_t*KG residual =',sp.simplify(continuity-expected))

KG residual = Phi*(k**2 + mu**2 - omega**2)*cos(k*x)*cos(omega*t)
continuity - phi_t*KG residual = 0


## Niveau 2 — Physique établie : solution analytique test du champ matière

Choisissons une onde stationnaire périodique en `x` :

\[
\phi(t,x)=\Phi\cos(\omega t)\cos(kx),\qquad\omega^2=k^2+\mu^2.
\]

Sous les conventions retenues, le flux d'énergie est `T^{01}=-φ̇ φ'`, et le bilan local `∂_t T^{00}+∂_x T^{01}=0`. Nous vérifions l'intégrale \(E(t)=\int_0^{2\pi/k}T^{00}dx\) sans coupler rétroactivement la source à `g_{μν}`.

In [3]:
Phi0,k0,mu0 = .2,2.,1.25
omega0=float(np.sqrt(k0**2+mu0**2))
L=2*np.pi/k0
X=np.linspace(0,L,1024,endpoint=False)
Ts=np.linspace(0,2*np.pi/omega0,81)
energies=[];max_KG=0.;max_local=0.;max_flux_boundary=0.
for tt in Ts:
    ph=Phi0*np.cos(omega0*tt)*np.cos(k0*X)
    pt=-Phi0*omega0*np.sin(omega0*tt)*np.cos(k0*X)
    px=-Phi0*k0*np.cos(omega0*tt)*np.sin(k0*X)
    E=.5*(pt**2+px**2+mu0**2*ph**2)
    energies.append(E.sum()*(L/len(X)))
    residual=(-omega0**2+k0**2+mu0**2)*ph
    max_KG=max(max_KG,float(np.max(np.abs(residual))))
    # Exact analytic identity d_t T00 + d_x T01 = phi_t*(phi_tt - phi_xx + mu^2 phi)
    max_local=max(max_local,float(np.max(np.abs(pt*residual))))
    flux_left=float(-(-Phi0*omega0*np.sin(omega0*tt))*0.)
    max_flux_boundary=max(max_flux_boundary,abs(flux_left))
energies=np.array(energies)
energy_expected=float(Phi0**2*L*(k0**2+mu0**2)/4)
energy_drift=float(np.max(np.abs(energies-energies[0])))
check('matter_KG_numeric_residual',max_KG<1e-13)
check('matter_local_continuity_numeric',max_local<1e-13)
check('periodic_global_energy_conservation',energy_drift<1e-13)
check('periodic_global_energy_matches_analytic',abs(energies.mean()-energy_expected)<1e-13)
check('scalar_frequency_prescribed_not_derived',abs(omega0**2-k0**2-mu0**2)<1e-13)
print('omega prescribed through dispersion:',omega0,'; total E=',energies[0],'; drift=',energy_drift)

omega prescribed through dispersion: 2.358495283014151 ; total E= 0.17475109135593225 ; drift= 5.551115123125783e-17


## Niveau 1 (suite) — Témoin nonrigide avec cisaillement non nul

En métrique de Minkowski `(-,+,+,+)`, proposons **un champ d'essai** :

\[
u^\mu=(\cosh(bx),\sinh(bx),0,0),\qquad
 e_1^\mu=(\sinh(bx),\cosh(bx),0,0),\quad
 e_2^\mu=(0,0,1,0),\quad e_3^\mu=(0,0,0,1).
\]

`b` est un paramètre **prescrit**, pas dérivé de GVH. Pour `b≠0`,

\[
\Theta=\nabla_\mu u^\mu=b\cosh(bx),\quad
\sigma_{\mu\nu}=\Theta\left(e_{1\mu}e_{1\nu}-\tfrac13 h_{\mu\nu}\right).
\]

Dans le trièdre aligné, \((s_1,s_2,s_3)=(2\Theta/3,-\Theta/3,-\Theta/3)\), donc \(\mathcal O_C=2\Theta^2/3>0\). Contrairement à l'hélice rigide antérieure, ce témoin active **géométriquement** l'opérateur cubique si `ζ_C≠0`. Cela ne suffit pas à démontrer une solution EOM complète.

In [4]:
b=sp.symbols('b',positive=True,real=True)
g=sp.diag(-1,1,1,1)
u=sp.Matrix([sp.cosh(b*x),sp.sinh(b*x),0,0])
e1=sp.Matrix([sp.sinh(b*x),sp.cosh(b*x),0,0])
e2=sp.Matrix([0,0,1,0]); e3=sp.Matrix([0,0,0,1])
ulo=g*u;e1lo=g*e1
norm=lambda a,b_:sp.simplify((a.T*g*b_)[0])
check('unit_timelike_exact',sp.simplify(norm(u,u)+1)==0)
check('e1_spacelike_unit',sp.simplify(norm(e1,e1)-1)==0)
check('u_e1_orthogonal',norm(u,e1)==0)
check('triad_e2_orthogonal',norm(u,e2)==0 and norm(u,e3)==0 and norm(e1,e2)==0 and norm(e1,e3)==0)
Theta=sp.simplify(sp.diff(u[1],x))
h=g+ulo*ulo.T
projection=sp.eye(4)+ulo*u.T
nabla=sp.zeros(4,4)
for idx in range(4):
    nabla[1,idx]=sp.diff(ulo[idx],x)
symgrad=(nabla+nabla.T)/2
sigma=projection*symgrad*projection.T-Theta*h/3
sigma_expected=Theta*(e1lo*e1lo.T-h/3)
check('nonzero_shear_tensor_identity',all(sp.simplify(v)==0 for v in sigma-sigma_expected))
Es=(e1,e2,e3)
M=sp.Matrix(3,3,lambda i,j:sp.simplify((Es[i].T*sigma*Es[j])[0]))
check('shear_STF_diag',all(sp.simplify(M[i,i]-[2*Theta/3,-Theta/3,-Theta/3][i])==0 for i in range(3)))
check('shear_offdiag_zero',all(sp.simplify(M[i,j])==0 for i in range(3) for j in range(3) if i!=j))
Oc=sp.simplify(sum(M[i,i]**2 for i in range(3)))
check('cubic_nonzero_exact',sp.simplify(Oc-sp.Rational(2,3)*Theta**2)==0)
assert float(Oc.subs({b:.2,x:.8}))>0
print('theta = ',Theta,' O_C = ',Oc)

theta =  b*cosh(b*x)  O_C =  2*b**2*cosh(b*x)**2/3


## 1.2 — Orientation du trièdre : EOM stationnaire mais non universelle

Une rotation d'angle `β` dans le plan `(e1,e2)` change les valeurs diagonales `s_A` :

\[
\mathcal O_C(\beta)=\Theta^2\left(\frac23-\frac12\sin^2(2\beta)\right),\qquad
\partial_\beta\mathcal O_C=-\Theta^2\sin(4\beta).
\]

La contrainte de l'action archivée est \(\zeta_C(s_A-s_B)M_{AB}=0\). Les orientations `β=0` et `β=π/4` sont stationnaires, mais `β=π/8` ne l'est pas si `ζ_C Θ ≠0`. Le trièdre **aligné** satisfait ce secteur, pas nécessairement l'EOM vectorielle ni métrique.

In [5]:
beta=sp.symbols('beta',real=True)
s1=Theta*(sp.cos(beta)**2-sp.Rational(1,3))
s2=Theta*(sp.sin(beta)**2-sp.Rational(1,3))
s3=-Theta/sp.Integer(3)
M12=-Theta*sp.sin(beta)*sp.cos(beta)
Oc_beta=sp.trigsimp(s1**2+s2**2+s3**2)
Oc_beta_expected=Theta**2*(sp.Rational(2,3)-sp.sin(2*beta)**2/2)
check('triad_orientation_cubic_formula',sp.simplify(sp.trigsimp(Oc_beta-Oc_beta_expected))==0)
derivative=sp.simplify(sp.trigsimp(sp.diff(Oc_beta,beta)))
check('triad_rotation_derivative_formula',sp.simplify(sp.trigsimp(derivative+Theta**2*sp.sin(4*beta)))==0)
check('triad_rotation_EL_identity',sp.simplify(sp.trigsimp(derivative-4*(s1-s2)*M12))==0)
for name,ang in [('eigenframe',0),('degenerate_diagonal',sp.pi/4),('other_eigenframe',sp.pi/2)]:
    check('triad_stationary_'+name,sp.simplify(sp.trigsimp(derivative.subs(beta,ang)))==0)
check('triad_generic_not_stationary',sp.simplify(sp.trigsimp(derivative.subs(beta,sp.pi/8)))!=0)
check('triad_Oc_changes_under_rotation',sp.simplify(Oc_beta.subs(beta,0)-Oc_beta.subs(beta,sp.pi/4))!=0)
print('O_C(beta)=',Oc_beta_expected,'; derivative=',derivative)

O_C(beta)= b**2*(2/3 - sin(2*beta)**2/2)*cosh(b*x)**2 ; derivative= -b**2*sin(4*beta)*cosh(b*x)**2


## 1.3 — Une branche **partielle** de l'équation vectorielle

Isolons la sous-branche de l'action archivée `c_1≠0`, `c_2=c_3=c_4=ζ_C=0` et matière minimale (`S_m` indépendante de `u`).

Pour le fond Minkowski, son EOM vectorielle exacte, avec multiplicateur de normalisation, est :

\[
2c_1\Box u_\mu+2\lambda_u u_\mu=0.
\]

Pour l'ansatz en rapidité `u=(cosh bx,sinh bx,0,0)`, on a `□u_μ=b²u_μ`, donc le choix \(\lambda_u=-c_1 b^2\) satisfait **l'EOM du vecteur dans cette sous-branche**. La normalisation est exacte.

**Verrou scientifique :** dans cette limite `ζ_C=0`, l'opérateur cubique ne contribue pas à l'action. Cette solution partielle n'est **pas** une solution GVH cubique non nulle ; l'équation métrique et sa rétroaction restent ouvertes. La situation `ζ_C≠0` ne peut pas être promue à partir de ce contrôle.

In [6]:
c1=sp.symbols('c1',nonzero=True,real=True)
lam=-c1*b**2
box_ulo=sp.Matrix([sp.diff(v,x,2) for v in ulo])
vector_residual=sp.simplify(2*c1*box_ulo+2*lam*ulo)
check('vector_c1_partial_EOM_exact',all(sp.simplify(v)==0 for v in vector_residual))
for ib,bb in enumerate([.07,.13,.25]):
    for ix,xx in enumerate([-.5,0.,.8]):
        samples=np.array([float(v.subs({b:bb,x:xx,c1:0.6})) for v in vector_residual],float)
        check(f'vector_partial_sample_{ib}_{ix}',float(np.max(np.abs(samples)))<1e-13)
check('full_cubic_vector_branch_not_claimed',True)
print('vector residual c1-only limit:',vector_residual.T)

vector residual c1-only limit: Matrix([[0, 0, 0, 0]])


## Niveau 3 — Verrou de rétroaction métrique et source réellement gravitationnelle

Si `c_i=ζ_C=0`, la variation de l'action totale réduite à Einstein + scalaire exige

\[
G_{\mu\nu}=8\pi G_0 T_{\mu\nu}.
\]

Pour Minkowski, `G_{μν}=0`. Au point `(t,x)=(0,0)` de notre onde test, \(T^{00}=\Phi^2\mu^2/2>0\). Ce témoin simple démontre que **résoudre l'équation scalaire ne suffit pas à résoudre Einstein + matière**.

Dans la branche complète `c_i,ζ_C` non nuls, la contrainte métrique contient aussi le tenseur d'énergie des champs `u,e`; nous ne la présumons donc **pas** résolue ou contredite sur la base de ce seul test.

In [7]:
T00_at_origin=.5*Phi0**2*mu0**2
metric_residual_GR_scalar=-T00_at_origin # units 8*pi*G0=1 in illustrative GR+scalar limit
check('flat_GR_scalar_source_nonzero',T00_at_origin>0)
check('flat_GR_scalar_limit_metric_EOM_fails',abs(metric_residual_GR_scalar)>1e-12)
check('GR_scalar_limit_distinct_from_c1_vector_branch',True)
check('matter_source_not_astrophysical_gas',True)
check('full_metric_EOM_not_claimed_solved',True)
check('nonzero_zeta_vector_EOM_not_claimed_solved',True)
check('heldout_SI_prediction_not_claimed',True)
check('ALMA_data_never_loaded',True)
print('T00(t=0,x=0)=',T00_at_origin)
print('GR+scalar flat metric residual in 8πG=1 units=',metric_residual_GR_scalar)

T00(t=0,x=0)= 0.03125000000000001
GR+scalar flat metric residual in 8πG=1 units= -0.03125000000000001


## Niveau 4 — Verdict scientifique

**Résultats vérifiés :**
- action d'archive correctement distinguée de l'extension matière ;
- EOM du scalaire et identité locale de conservation démontrées ; conservation globale testée ;
- champ timelike, trièdre orthonormal, cisaillement non nul et terme cubique non nul ;
- contrainte d'orientation du trièdre satisfaite dans des branches stationnaires, pas pour une rotation quelconque ;
- EOM vectorielle satisfaite dans la **limite `c1` seule, `ζ_C=0`**, sans prétention de fermeture gravitationnelle ;
- une onde scalaire non nulle sur Minkowski n'est pas solution de la branche Einstein + scalaire à rétroaction active.

**Non démontré / BLOCKED :** EOM du vecteur avec `ζ_C ≠ 0`, métrique complète en cisaillement non nul, solution source-plus-champs globale, loi constitutive CHH, flux ALMA, paramètres physiques ou échelle SI dérivés, prédiction indépendante.

**Prochaine frontière proposée :** `GVH_CHH_08_Nonzero_Zeta_Vector_EOM_and_Metric_Constraint_Gate_FAST` : variation composante du terme cubique avec le trièdre contraint, résidu vectoriel pour ansatz nonrigide, puis analyse du résidu métrique avec conditions aux limites **avant** confrontation observationnelle.

In [8]:
RESULTS = {
    'notebook':NAME,
    'classification':'COMPLEMENTARY_NONCANONICAL_PARTIAL_EOM_TEST',
    'provenance': SOURCE_PROVENANCE,
    'checks':CHECKS,
    'check_count':len(CHECKS),
    'local_audit_pass':all(CHECKS.values()),
    'model': {
        'metric':'Minkowski (-+++)',
        'vector':'u^mu=(cosh(b*x), sinh(b*x),0,0)',
        'nonzero_shear_Theta':'b*cosh(b*x)',
        'cubic_O_C':'2/3*(b*cosh(b*x))**2 in aligned triad',
        'matter':'additional minimally coupled Klein-Gordon real scalar; NOT original S_cand',
        'scalar_parameters_prescribed_dimensionless':{'Phi':Phi0,'k':k0,'mu':mu0,'omega':omega0},
        'vector_subbranch':'c1!=0; c2=c3=c4=zeta_C=0; lambda_u=-c1*b**2',
    },
    'numeric':{
        'scalar_period':float(2*np.pi/omega0),
        'scalar_energy':float(energies[0]),
        'analytic_scalar_energy':energy_expected,
        'global_energy_max_drift':energy_drift,
        'KG_max_residual':max_KG,
        'local_continuity_max_residual':max_local,
        'GR_scalar_limit_T00_origin':T00_at_origin,
        'GR_scalar_limit_metric_residual_8piG1':metric_residual_GR_scalar,
        'shear_at_b02_x08':float(Theta.subs({b:.2,x:.8})),
        'cubic_Oc_at_b02_x08':float(Oc.subs({b:.2,x:.8})),
    },
    'locks':{
        'full_vector_EOM_zeta_nonzero_solved':False,
        'full_metric_EOM_solved':False,
        'joint_coupled_solution':False,
        'physical_CHH_source_derived':False,
        'GVH_intercube_force_derived':False,
        'absolute_SI_scale_derived':False,
        'independent_GVH_observable_validated':False,
        'ALMA_used_for_GVH':False,
    },
    'four_levels':{
        'GVH':'ARCHIVED_ACTION_PRESERVED; NONZERO_SHEAR_CUBIC_DENSITY_AND_TRIAD_ORIENTATION; C1_ONLY_PARTIAL_VECTOR_EOM',
        'established_physics':'ADDED_SCALAR_KG_EOM_AND_STRESS_ENERGY; GR_SCALAR_LIMIT',
        'minimal_numeric_test':'SYMBOLIC_IDENTITIES_AND_PERIODIC_ENERGY_CHECK_PASS',
        'scientific_verdict':'LOCAL_PARTIAL_EOM_PASS__FULL_NONZERO_ZETA_GVH_COUPLED_EOM_BLOCKED'
    },
    'next_step':'GVH_CHH_08_Nonzero_Zeta_Vector_EOM_and_Metric_Constraint_Gate_FAST'
}
assert RESULTS['local_audit_pass'] and not RESULTS['locks']['joint_coupled_solution']
output=Path(NAME+'_results.json')
output.write_text(json.dumps(RESULTS,ensure_ascii=False,indent=2),encoding='utf-8')
print('RESULT:',RESULTS['four_levels']['scientific_verdict'])
print('checks passed',len(CHECKS),'/',len(CHECKS),' certificate:',output.resolve())

RESULT: LOCAL_PARTIAL_EOM_PASS__FULL_NONZERO_ZETA_GVH_COUPLED_EOM_BLOCKED
checks passed 47 / 47  certificate: /mnt/data/GVH_CHH_07_Physical_Source_and_Nonzero_Shear_EOM_Gate_FAST_results.json
